# 03 — Financial Analysis & Bidding Strategy
**Solar Power Forecasting | System 2107, Arbuckle CA**

**Prerequisites:**
1. Run `01_data_pipeline.py`
2. Run `02_model_comparison.ipynb` fully (needs `results`, `df_test_feat`, `y_test`)

This notebook:
- Downloads CAISO Day-Ahead LMP prices
- Computes MWOL for every model
- Produces financial comparison plots
- Finds the optimal bid discount strategy


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import requests, zipfile, io, time
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ── Check that model_comparison results are available ─────────────────────────
try:
    results
    print(f"results dict loaded: {list(results.keys())}")
except NameError:
    raise RuntimeError(
        "Run 02_model_comparison.ipynb first — this notebook needs the "
        "`results`, `df_test_feat`, and `y_test` variables."
    )


## 12. LMP Data Loading & Financial Analysis
**Locational Marginal Price (LMP)** is the hourly electricity price at the grid node.
It determines revenue earned per MWh of solar energy delivered to the grid.

**Data source:** CAISO OASIS API (pure `requests`, Python 3.10 compatible)  
- Node: `TH_NP15_GEN-APND` (NP15 trading hub — Northern California, covers Arbuckle area)  
- Resolution: hourly, local Pacific time  
- Coverage: 2023-01-23 onward (~10–12 min download in weekly chunks)

**MWOL formula (from user specification):**
```
penalty = max(0, LMP × (predicted_MWh − actual_MWh))
```
Covers both:
- Positive LMP + over-prediction → buy shortfall at market price
- Negative LMP + under-prediction → selling less when price is negative (avoided cost)

**Financial Efficiency:**
`FE (%) = (1 − Total_MWOL / Total_Potential_Revenue) × 100`


In [ ]:
# ── Download LMP from CAISO OASIS API (pure requests, no gridstatus) ─────────
# Uses the CAISO OASIS PRC_LMP query — no external dependencies beyond requests.
# Node  : TH_NP15_GEN-APND  (NP15 trading hub, Northern California)
# Market: DAM (Day-Ahead Market)
# Chunks: 30-day windows with 2 s pause — ~10-12 min total download

import requests, zipfile, io, time

def fetch_caiso_lmp_oasis(start_date, end_date,
                           node="TH_NP15_GEN-APND",
                           market="DAM",
                           chunk_days=30):
    """Download hourly Day-Ahead LMP from CAISO OASIS.

    Returns a DataFrame with columns:
      'Interval Start' (tz-naive Pacific), 'LMP' ($/MWh)
    """
    base = "http://oasis.caiso.com/oasisapi/SingleZip"
    cur  = pd.Timestamp(start_date)
    end  = pd.Timestamp(end_date)
    records = []

    print(f"Fetching CAISO LMP {start_date} → {end_date}  node={node}")
    while cur < end:
        nxt = min(cur + pd.Timedelta(days=chunk_days), end)
        params = {
            "queryname":     "PRC_LMP",
            "startdatetime": cur.strftime("%Y%m%dT00:00-0000"),  # UTC
            "enddatetime":   nxt.strftime("%Y%m%dT00:00-0000"),
            "version":       1,
            "market_run_id": market,
            "node":          node,
            "resultformat":  6,   # CSV inside ZIP
        }
        try:
            r = requests.get(base, params=params, timeout=90)
            r.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                csv_name = next(n for n in z.namelist() if n.endswith(".csv"))
                chunk = pd.read_csv(z.open(csv_name))

            # Keep only the composite LMP row (excludes energy/loss/congestion)
            chunk = chunk[chunk["LMP_TYPE"] == "LMP"].copy()

            # OASIS returns GMT timestamps — convert to tz-naive Pacific
            chunk["Interval Start"] = (
                pd.to_datetime(chunk["INTERVALSTARTTIME_GMT"], utc=True)
                  .dt.tz_convert("US/Pacific")
                  .dt.tz_localize(None)
            )
            chunk = chunk.rename(columns={"MW": "LMP"})[["Interval Start","LMP"]]
            records.append(chunk)
            print(f"  {cur.date()} → {nxt.date()} : {len(chunk)} rows")
        except Exception as e:
            print(f"  WARNING {cur.date()} → {nxt.date()} : {e}")
        cur = nxt
        time.sleep(2)   # respect CAISO rate limit

    if not records:
        raise RuntimeError("No LMP data returned — check node name or date range")

    lmp_raw = (pd.concat(records, ignore_index=True)
                 .drop_duplicates("Interval Start")
                 .sort_values("Interval Start")
                 .reset_index(drop=True))

    print(f"\nTotal rows : {len(lmp_raw):,}")
    print(f"Date range : {lmp_raw['Interval Start'].min()} → "
          f"{lmp_raw['Interval Start'].max()}")
    print(f"LMP stats  : mean=${lmp_raw['LMP'].mean():.2f}  "
          f"min=${lmp_raw['LMP'].min():.2f}  "
          f"max=${lmp_raw['LMP'].max():.2f} /MWh")
    return lmp_raw

# ── Run the download ──────────────────────────────────────────────────────────
# LMP starts 2023-01-23; we cover the full val + test window
lmp_df = fetch_caiso_lmp_oasis(
    start_date = "2024-01-01",   # covers val (Jan-May) and test (Jun-Oct)
    end_date   = "2024-11-01",
    node       = "TH_NP15_GEN-APND",
)

# Alias for downstream cells
lmp_clean = lmp_df.copy()
print("\nlmp_clean ready.")
print(lmp_clean.head(3))


In [ ]:
# ── lmp_clean is already prepared in Cell 30 ─────────────────────────────────
# Just display a quick overview plot of the LMP data.

import matplotlib as mpl
mpl.rcParams.update({"font.size":16,"axes.titlesize":20,"axes.labelsize":18,
                     "xtick.labelsize":14,"ytick.labelsize":14,
                     "axes.spines.top":False,"axes.spines.right":False})

print(f"lmp_clean shape : {lmp_clean.shape}")
print(f"Date range      : {lmp_clean['Interval Start'].min()} → "
      f"{lmp_clean['Interval Start'].max()}")
print(f"LMP stats       : mean=${lmp_clean['LMP'].mean():.2f}  "
      f"min=${lmp_clean['LMP'].min():.2f}  "
      f"max=${lmp_clean['LMP'].max():.2f} /MWh")
print(lmp_clean.head(3))

# ── LMP overview plot ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(22, 6))
ax.plot(lmp_clean["Interval Start"], lmp_clean["LMP"],
        linewidth=0.7, color="#2196F3", alpha=0.85)
ax.axhline(lmp_clean["LMP"].mean(), color="#FF5722", linewidth=1.5,
           linestyle="--", label=f"Mean ${lmp_clean['LMP'].mean():.2f}/MWh")
ax.set_title("CAISO Day-Ahead LMP — NP15 Hub (2024)", pad=12)
ax.set_ylabel("LMP ($/MWh)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.legend(fontsize=14)
plt.tight_layout()
plt.savefig("figF0_lmp_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figF0_lmp_overview.png")


## 13. Financial Metric Functions

In [ ]:
# ── Financial metric functions (matching user specification) ─────────────────

def meter_to_hourly_mwh(df_15min, time_col="measured_on"):
    """Convert 15-min kW readings → hourly MWh.
    
    Each 15-min interval: energy = power_kW × 0.25h × (1 MWh/1000 kWh)
    Then resample to hourly by summing four 15-min intervals.
    Returns a DataFrame indexed by hour with numeric columns in MWh.
    """
    df = df_15min.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    # Strip timezone if present (keep tz-naive to match LMP)
    if df[time_col].dt.tz is not None:
        df[time_col] = df[time_col].dt.tz_localize(None)
    df = df.sort_values(time_col)
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # kW × 0.25h × 0.001 = MWh per 15-min interval
    df[num_cols] = df[num_cols] * 0.001 * 0.25
    hourly = df.set_index(time_col)[num_cols].resample("1h").sum(min_count=1)
    return hourly.reset_index()


def compute_market_weighted_loss(actuals_15min, preds_15min, lmp_df,
                                  time_col="measured_on"):
    """Market Weighted Opportunity Loss (MWOL) — user specification formula.

    MWOL penalty per hour = max(0, LMP × (predicted_MWh − actual_MWh))

    This covers two harmful cases:
      1. Positive LMP + over-prediction → committed to sell more than produced,
         must buy shortfall at market price.
      2. Negative LMP + under-prediction → by producing more than committed,
         we export at a negative price, incurring a cost (avoided by under-committing).

    Potential revenue = max(0, LMP × actual_MWh)  (revenue from perfect forecast)

    Parameters
    ----------
    actuals_15min : DataFrame with [time_col, actual_power_kw]
    preds_15min   : DataFrame with [time_col, predicted_power_kw]
    lmp_df        : DataFrame with ['Interval Start', 'LMP']
    time_col      : name of the timestamp column in actuals/preds

    Returns
    -------
    dict with total MWOL ($), mean daily MWOL ($/day), financial efficiency (%),
    daily series, and the detailed hourly DataFrame for inspection.
    """
    # 1. Convert 15-min kW → hourly MWh
    actuals_h = meter_to_hourly_mwh(actuals_15min, time_col=time_col)
    preds_h   = meter_to_hourly_mwh(preds_15min,   time_col=time_col)

    # 2. Align with LMP on 'Interval Start'
    lmp_work = lmp_df.copy()
    lmp_work["Interval Start"] = pd.to_datetime(lmp_work["Interval Start"])
    if lmp_work["Interval Start"].dt.tz is not None:
        lmp_work["Interval Start"] = (lmp_work["Interval Start"]
                                       .dt.tz_convert("US/Pacific")
                                       .dt.tz_localize(None))

    # Merge actuals and predictions onto LMP timestamps
    val_col  = [c for c in actuals_h.columns if c != time_col][0]  # e.g. 'power_kw'

    # Rename before merging so both sides have distinct column names,
    # avoiding the pandas suffixing that causes KeyError when names collide.
    actuals_h = actuals_h.rename(columns={val_col: "actual_mwh"})
    preds_h   = preds_h.rename(columns={val_col: "pred_mwh"})
    act_col   = "actual_mwh"
    pred_col  = "pred_mwh"

    df = (lmp_work
          .merge(actuals_h, left_on="Interval Start", right_on=time_col, how="left")
          .merge(preds_h,   left_on="Interval Start", right_on=time_col, how="left"))

    # Drop hours missing any of the three required values
    df = df.dropna(subset=["LMP", act_col, pred_col])

    # 3. Compute penalty and potential revenue per hour
    # Penalty: positive LMP × over-prediction OR negative LMP × under-prediction
    df["mwol_penalty"]      = np.maximum(0, df["LMP"] * (df[pred_col] - df[act_col]))
    df["potential_revenue"] = np.maximum(0, df["LMP"] * df[act_col])

    # 4. Daily aggregation
    daily = df.groupby(df["Interval Start"].dt.date).agg(
        mwol_penalty=("mwol_penalty",      "sum"),
        potential_revenue=("potential_revenue", "sum"),
    )
    daily.index = pd.to_datetime(daily.index)

    # 5. Summary metrics
    total_loss      = daily["mwol_penalty"].sum()
    total_potential = daily["potential_revenue"].sum()
    mean_daily_mwol = daily["mwol_penalty"].mean()
    efficiency      = (1 - total_loss / total_potential) * 100 if total_potential > 0 else 0.0

    return {
        "Total MWOL ($)":          total_loss,
        "Mean Daily MWOL ($/day)": mean_daily_mwol,
        "Financial Efficiency (%)": efficiency,
        "Daily_MWOL_Series":       daily["mwol_penalty"],
        "Daily_Potential":         daily["potential_revenue"],
        "Detailed_DF":             df,
    }


def financial_efficiency_pct(actuals_15min, preds_15min, lmp_df,
                               time_col="measured_on"):
    """Convenience wrapper returning only Financial Efficiency (%)."""
    r = compute_market_weighted_loss(actuals_15min, preds_15min, lmp_df, time_col)
    return r["Financial Efficiency (%)"]


print("Financial functions defined:")
print("  meter_to_hourly_mwh()          → converts 15-min kW → hourly MWh")
print("  compute_market_weighted_loss()  → MWOL, daily series, FE%")
print("  financial_efficiency_pct()      → quick FE% wrapper")


## 14. Compute Financial Metrics for All Models

In [ ]:
# ── Prepare actuals DataFrame ────────────────────────────────────────────────
# Explicit single column named 'power_kw' to avoid any naming collisions
# inside compute_market_weighted_loss when the same df is passed twice.

actuals_15min = pd.DataFrame({
    "measured_on": df_test_feat.index,
    "power_kw":    y_test.values,          # actual power, explicitly named
})

print("Computing MWOL for all models...")
print(f"Actuals shape : {actuals_15min.shape}")
print(f"LMP coverage  : {lmp_clean['Interval Start'].min().date()} → "
      f"{lmp_clean['Interval Start'].max().date()}")
print()

financial_results = {}
for name, r in results.items():
    # Build predictions DataFrame — same single-column structure
    preds_15min = pd.DataFrame({
        "measured_on": df_test_feat.index,
        "power_kw":    r["preds"],          # predicted power, same column name
    })

    res = compute_market_weighted_loss(
        actuals_15min,
        preds_15min,
        lmp_clean,
        time_col="measured_on",
    )

    financial_results[name] = {
        "mwol":       res["Total MWOL ($)"],
        "mean_daily": res["Mean Daily MWOL ($/day)"],
        "fe_pct":     res["Financial Efficiency (%)"],
        "daily":      res["Daily_MWOL_Series"],
        "daily_pot":  res["Daily_Potential"],
        "rmse_pct":   r["rmse_pct"],
    }

    rmse_day = r.get("rmse_kw", float("nan"))
    rmse_all = r.get("rmse_kw_all", float("nan"))
    print(f"  {name:<22}  "
          f"MWOL=${res['Total MWOL ($)']:>10,.2f}  "
          f"FE={res['Financial Efficiency (%)']:>7.2f}%  "
          f"RMSE%={r['rmse_pct']:.1f}%  "
          f"RMSE={rmse_day:.1f}kW(day)/{rmse_all:.1f}kW(all)")

# ── Backfill rmse_kw for all models (computed here where preds are available) ─
# ML model results were built before rmse_absolute() existed, so rmse_kw=NaN.
# Recompute here using the test actuals and stored predictions.
for name, r in results.items():
    if np.isnan(r.get("rmse_kw", np.nan)):
        preds = r["preds"]
        dt    = y_test.values > 0
        if dt.sum() > 0:
            results[name]["rmse_kw"]     = float(np.sqrt(np.mean(
                (preds[dt] - y_test.values[dt]) ** 2)))
            results[name]["rmse_kw_all"] = float(np.sqrt(np.mean(
                (preds    - y_test.values)     ** 2)))

# ── Perfect forecast revenue ──────────────────────────────────────────────────
# Pass actuals as both actual and predicted — MWOL = 0 for every hour,
# potential_revenue = sum of all positive LMP × actual_MWh intervals.
# We compute this separately to get the denominator for FE%.
perfect_res    = compute_market_weighted_loss(
    actuals_15min, actuals_15min, lmp_clean, time_col="measured_on"
)
perfect_profit = perfect_res["Daily_Potential"].sum()
print(f"\nPerfect forecast potential revenue: ${perfect_profit:,.2f}")

# ── Summary table ─────────────────────────────────────────────────────────────
fin_summary = pd.DataFrame({
    name: {
        "MWOL ($)":              v["mwol"],
        "Mean Daily MWOL ($/d)": v["mean_daily"],
        "Revenue Lost (%)":      v["mwol"] / perfect_profit * 100
                                 if perfect_profit > 0 else np.nan,
        "Financial Eff. (%)":    v["fe_pct"],
        "RMSE%":                 v["rmse_pct"],
        "RMSE kW (daytime)":     results[name].get("rmse_kw", np.nan),
        "RMSE kW (all hrs)":     results[name].get("rmse_kw_all", np.nan),
    }
    for name, v in financial_results.items()
}).T.sort_values("MWOL ($)")

print("\n" + "="*72)
print("  FINANCIAL COMPARISON — Test Set (Jun–Oct 2024)")
print(f"  Perfect forecast revenue: ${perfect_profit:,.2f}")
print("="*72)
print(fin_summary.round(3).to_string())
print("="*72)


## 15. Financial Comparison Plots

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    "font.size":         18,
    "axes.titlesize":    22,
    "axes.labelsize":    20,
    "xtick.labelsize":   17,
    "ytick.labelsize":   17,
    "legend.fontsize":   16,
    "axes.linewidth":    1.4,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

FIN_COLORS = {
    "LightGBM":          "#2196F3",
    "XGBoost":           "#FF5722",
    "RandomForest":      "#4CAF50",
    "Ridge":             "#9C27B0",
    "LinearReg":         "#00BCD4",  # cyan
    "MLP":               "#E91E63",
    "Persistence":       "#607D8B",
    "Climatology":       "#795548",
    "SmartPersistence":  "#009688",
    "Stacking":          "#F44336",  # red — the ensemble
}
fin_names = list(financial_results.keys())

# ── Fig F0: LMP overview (already saved in Cell 31) ──────────────────────────

# ── Fig F1: Total MWOL bar chart ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 8))
mwol_vals = [financial_results[n]["mwol"] for n in fin_names]
bars = ax.bar(fin_names, mwol_vals,
              color=[FIN_COLORS.get(n, "#888") for n in fin_names],
              alpha=0.88, edgecolor="white", width=0.6)
ax.bar_label(bars, labels=[f"${v:,.0f}" for v in mwol_vals],
             fontsize=16, padding=5, fontweight="bold")
ax.set_title("Total Market Weighted Opportunity Loss (MWOL) — Test Set\n"
             "Lower = less revenue lost due to forecast error", pad=14)
ax.set_ylabel("Revenue Lost ($)")
ax.set_ylim(0, max(mwol_vals) * 1.22)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=25, ha="right")
plt.tight_layout()
plt.savefig("figF1_mwol_total.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figF1_mwol_total.png")

# ── Fig F2: Financial Efficiency bar chart ────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 8))
fe_vals = [financial_results[n]["fe_pct"] for n in fin_names]
bars2 = ax.bar(fin_names, fe_vals,
               color=[FIN_COLORS.get(n, "#888") for n in fin_names],
               alpha=0.88, edgecolor="white", width=0.6)
ax.bar_label(bars2, labels=[f"{v:.2f}%" for v in fe_vals],
             fontsize=16, padding=5, fontweight="bold")
ax.axhline(100, color="black", linewidth=1.2, linestyle="--",
           alpha=0.4, label="Perfect = 100%")
ax.set_title("Financial Efficiency  [FE% = (1 − MWOL / Potential Revenue) × 100]\n"
             "Higher = more potential revenue captured", pad=14)
ax.set_ylabel("Financial Efficiency (%)")
ax.set_ylim(min(0, min(fe_vals)) - 5, 110)
ax.legend(fontsize=14)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=25, ha="right")
plt.tight_layout()
plt.savefig("figF2_financial_efficiency.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figF2_financial_efficiency.png")

# ── Fig F3: RMSE% vs MWOL scatter ─────────────────────────────────────────────
# Key question: does lower RMSE% translate directly to lower money loss?
fig, ax = plt.subplots(figsize=(16, 10))
for name in fin_names:
    ax.scatter(financial_results[name]["rmse_pct"],
               financial_results[name]["mwol"],
               color=FIN_COLORS.get(name, "#888"),
               s=300, zorder=5, edgecolors="white", linewidths=2)
    ax.annotate(f" {name}",
                xy=(financial_results[name]["rmse_pct"],
                    financial_results[name]["mwol"]),
                fontsize=14, color=FIN_COLORS.get(name, "#888"), va="center")
ax.set_title("RMSE% vs Market Weighted Opportunity Loss\n"
             "Lower-left corner = best (low error AND low money loss)", pad=14)
ax.set_xlabel("RMSE%  (forecast accuracy)")
ax.set_ylabel("MWOL ($)  (revenue lost)")
plt.tight_layout()
plt.savefig("figF3_rmse_vs_mwol.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figF3_rmse_vs_mwol.png")

# ── Fig F4: Daily MWOL time series ────────────────────────────────────────────
LINESTYLES_FIN = {
    "LightGBM":(0,()), "XGBoost":(0,(6,2)), "RandomForest":(0,(2,2)),
    "Ridge":(0,(4,2,1,2)), "MLP":(0,(1,1)),
    "LinearReg":        (0, (2, 1)),
    "Persistence":(0,(3,1,1,1,1,1)), "Climatology":(0,(5,5)),
    "SmartPersistence":(0,(3,1)),
    "Stacking":         (0,()),      # solid red — ensemble
}
fig, ax = plt.subplots(figsize=(26, 9))
for name in fin_names:
    d = financial_results[name]["daily"]
    ax.plot(d.index, d.values, color=FIN_COLORS.get(name,"#888"),
            linewidth=1.8, linestyle=LINESTYLES_FIN.get(name,(0,())),
            alpha=0.85, label=name)
ax.set_title("Daily Market Weighted Opportunity Loss — Test Set (Jun–Oct 2024)",
             pad=14)
ax.set_ylabel("Daily MWOL ($)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.legend(ncol=2, framealpha=0.4, loc="upper left")
plt.tight_layout()
plt.savefig("figF4_daily_mwol.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figF4_daily_mwol.png")

# ── Fig F5: Daily Financial Efficiency time series ────────────────────────────
fig, ax = plt.subplots(figsize=(26, 9))
for name in fin_names:
    d   = financial_results[name]["daily"]
    pot = financial_results[name]["daily_pot"].reindex(d.index).fillna(0)
    fe_daily = np.where(pot > 0, (1 - d.values / pot.values) * 100, np.nan)
    ax.plot(d.index, fe_daily,
            color=FIN_COLORS.get(name,"#888"), linewidth=1.8,
            linestyle=LINESTYLES_FIN.get(name,(0,())),
            alpha=0.85, label=name)
ax.axhline(100, color="black", linewidth=1.2, linestyle="--", alpha=0.3,
           label="Perfect = 100%")
ax.set_title("Daily Financial Efficiency — Test Set (Jun–Oct 2024)", pad=14)
ax.set_ylabel("Financial Efficiency (%)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.legend(ncol=2, framealpha=0.4, loc="lower left")
plt.tight_layout()
plt.savefig("figF5_daily_fe.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figF5_daily_fe.png")

# ── Fig F6: Monthly MWOL grouped bar chart ────────────────────────────────────
fig, ax = plt.subplots(figsize=(22, 9))
months   = sorted(set(financial_results[fin_names[0]]["daily"].index.to_period("M")))
x        = np.arange(len(months))
n_models = len(fin_names)
width    = 0.8 / n_models

for i, name in enumerate(fin_names):
    d = financial_results[name]["daily"]
    monthly = [d[d.index.to_period("M") == m].sum() for m in months]
    ax.bar(x + i*width - 0.4 + width/2, monthly, width,
           color=FIN_COLORS.get(name,"#888"), alpha=0.88,
           edgecolor="white", label=name)

ax.set_xticks(x)
ax.set_xticklabels([str(m) for m in months], rotation=25, ha="right")
ax.set_title("Monthly MWOL by Model — Test Set", pad=14)
ax.set_ylabel("Monthly MWOL ($)")
ax.legend(ncol=2, framealpha=0.4)
plt.tight_layout()
plt.savefig("figF6_monthly_mwol.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figF6_monthly_mwol.png")

# ── Fig F7: Mean daily MWOL bar chart ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 8))
mean_daily = [financial_results[n]["mean_daily"] for n in fin_names]
bars7 = ax.bar(fin_names, mean_daily,
               color=[FIN_COLORS.get(n,"#888") for n in fin_names],
               alpha=0.88, edgecolor="white", width=0.6)
ax.bar_label(bars7, labels=[f"${v:,.2f}" for v in mean_daily],
             fontsize=15, padding=5, fontweight="bold")
ax.set_title("Mean Daily MWOL by Model  ($/day average)", pad=14)
ax.set_ylabel("Mean Daily MWOL ($/day)")
ax.set_ylim(0, max(mean_daily) * 1.22)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=25, ha="right")
plt.tight_layout()
plt.savefig("figF7_mean_daily_mwol.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figF7_mean_daily_mwol.png")


In [ ]:
# ── Final financial summary ───────────────────────────────────────────────────
print("=" * 85)
print("  FINANCIAL ANALYSIS SUMMARY — Test Set (Jun–Oct 2024)")
print(f"  Perfect forecast revenue : ${perfect_profit:>12,.2f}")
print("  MWOL = max(0, LMP × (predicted_MWh − actual_MWh)) summed hourly")
print("  FE%  = (1 − MWOL / Potential Revenue) × 100")
print("=" * 85)
print(fin_summary.round(3).to_string())
print()
print("  ACTUAL RMSE (kW) — Test Set:")
print(f"  {'Model':<22} {'RMSE kW (daytime)':>20} {'RMSE kW (all hrs)':>20} {'RMSE%':>10}")
print("  " + "-"*75)
for name in fin_summary.index:
    r = results.get(name, {})
    rk   = r.get("rmse_kw",     float("nan"))
    rka  = r.get("rmse_kw_all", float("nan"))
    rpct = r.get("rmse_pct",    float("nan"))
    print(f"  {name:<22} {rk:>18.2f} kW  {rka:>17.2f} kW  {rpct:>9.2f}%")
print()
print("  Notes:")
print("  • RMSE kW (daytime) : hours where actual production > 0 only")
print("  • RMSE kW (all hrs) : all 15-min intervals incl. night zeros")
print("  • Night zeros are easy to predict → all-hours RMSE is always lower")
print("  • Daytime RMSE is the operationally meaningful number")
print()

ml_names = [n for n in financial_results
            if results[n]["model"] is not None and n not in BASELINE_NAMES]

best_fin  = fin_summary.loc[ml_names, "MWOL ($)"].idxmin()
worst_fin = fin_summary.loc[ml_names, "MWOL ($)"].idxmax()
savings   = financial_results[worst_fin]["mwol"] - financial_results[best_fin]["mwol"]

best_bl   = min(BASELINE_NAMES, key=lambda n: financial_results[n]["mwol"]
                if n in financial_results else np.inf)
if best_bl in financial_results:
    bl_mwol = financial_results[best_bl]["mwol"]
    print(f"  Best  ML model  : {best_fin:<15}  MWOL=${financial_results[best_fin]['mwol']:,.2f}  "
          f"FE={financial_results[best_fin]['fe_pct']:.2f}%")
    print(f"  Worst ML model  : {worst_fin:<15}  MWOL=${financial_results[worst_fin]['mwol']:,.2f}  "
          f"FE={financial_results[worst_fin]['fe_pct']:.2f}%")
    print(f"  Best baseline   : {best_bl:<15}  MWOL=${bl_mwol:,.2f}")
    print(f"  Revenue saved (best ML vs worst ML): ${savings:,.2f}")
    print()
    print("  Financial Skill Score (FSS_fin = 1 − ML_MWOL / Baseline_MWOL):")
    for name in ml_names:
        fss = 1 - financial_results[name]["mwol"] / bl_mwol
        print(f"    {name:<15}  FSS_fin = {fss:+.4f}")
print("=" * 85)
print()
print("  ⚠ Note on Stacking: higher MWOL than individual tree models despite")
print("    lower RMSE% — the meta-learner was trained on val-set predictions")
print("    (Jan–May 2024 winter/spring) but tested on Jun–Oct 2024 summer.")
print("    The seasonal distribution shift causes it to over-predict summer peaks.")
print("    Retrain stacking meta-learner on a rolling basis for operational use.")


---

## 16. Minimising Penalty Cost — Optimal Bidding Strategy

### Why raw forecast output is not the optimal bid

The MWOL penalty formula is:
```
penalty = max(0, LMP × (bid - actual))
```

This is **asymmetric by LMP sign**:

| LMP sign | Over-bid (bid > actual) | Under-bid (bid < actual) |
|---|---|---|
| **Positive** (+) | ❌ Penalty = LMP × excess | ✅ No penalty (just lost revenue) |
| **Negative** (−) | ✅ No penalty | ❌ Penalty = |LMP| × shortfall |

**Key insight:** When LMP > 0, you should bid *conservatively* (below forecast).  
When LMP < 0, you should bid *aggressively* (above forecast or at zero).

### Three strategies compared

1. **Raw forecast** — submit the model prediction as-is
2. **Conservative bid** — bid at `forecast × (1 - safety_margin)` during positive-LMP hours
3. **LMP-aware optimal bid** — use a quantile/safety buffer sized by LMP magnitude:
   - High positive LMP → bid lower (risk of over-prediction is expensive)
   - Near-zero or negative LMP → bid closer to forecast (penalty is cheap)


In [ ]:
# ── Penalty minimisation analysis ────────────────────────────────────────────
# Uses the best ML model's predictions and the actual LMP data to demonstrate
# how bid adjustment reduces MWOL without sacrificing much potential revenue.

import matplotlib as mpl
mpl.rcParams.update({
    "font.size":18,"axes.titlesize":22,"axes.labelsize":20,
    "xtick.labelsize":17,"ytick.labelsize":17,"legend.fontsize":16,
    "axes.spines.top":False,"axes.spines.right":False,
})

# ── Identify best ML model ────────────────────────────────────────────────────
ml_names_fin = [n for n in financial_results
                if results[n]["model"] is not None or n == "Stacking"]
best_ml = min(ml_names_fin, key=lambda n: financial_results[n]["mwol"])
print(f"Best ML model for bid optimisation: {best_ml}")

# ── Build hourly analysis DataFrame ──────────────────────────────────────────
actuals_h = (pd.DataFrame({"measured_on": df_test_feat.index,
                             "power_kw": y_test.values})
               .set_index("measured_on")
               .resample("1h").sum() * 0.001 * 0.25)   # kW × 0.25h × 0.001 = MWh

preds_h = (pd.DataFrame({"measured_on": df_test_feat.index,
                           "power_kw": results[best_ml]["preds"]})
             .set_index("measured_on")
             .resample("1h").sum() * 0.001 * 0.25)

lmp_h = lmp_clean.set_index("Interval Start")["LMP"].reindex(actuals_h.index, method="nearest")

analysis = pd.DataFrame({
    "actual_mwh":  actuals_h["power_kw"],
    "forecast_mwh": preds_h["power_kw"],
    "lmp":          lmp_h.values,
}).dropna()

analysis["error_mwh"]      = analysis["forecast_mwh"] - analysis["actual_mwh"]
analysis["lmp_pos"]         = analysis["lmp"] > 0
analysis["over_predicted"]  = analysis["error_mwh"] > 0

# ── Strategy 1: Raw forecast ──────────────────────────────────────────────────
analysis["bid_raw"] = analysis["forecast_mwh"]

# ── Strategy 2: Fixed conservative margin ─────────────────────────────────────
# Bid at 90% of forecast during positive-LMP hours (always conservative).
# During negative-LMP hours keep the raw forecast (under-bidding is penalised).
SAFETY_MARGIN = 0.10   # 10% below forecast during positive-LMP hours
analysis["bid_conservative"] = np.where(
    analysis["lmp_pos"],
    analysis["forecast_mwh"] * (1 - SAFETY_MARGIN),
    analysis["forecast_mwh"]   # no adjustment when LMP negative
)

# ── Strategy 3: LMP-proportional safety margin ───────────────────────────────
# Scale the margin by how expensive over-prediction is relative to average LMP.
# High LMP → larger discount; near-zero LMP → almost no discount.
# Formula: margin = base_margin × clamp(LMP / lmp_75pct, 0, 1)
lmp_pos_vals = analysis.loc[analysis["lmp_pos"], "lmp"]
lmp_75th     = lmp_pos_vals.quantile(0.75)

lmp_weight = (analysis["lmp"].clip(lower=0) / max(lmp_75th, 1)).clip(0, 1)
analysis["bid_lmp_aware"] = np.where(
    analysis["lmp_pos"],
    analysis["forecast_mwh"] * (1 - SAFETY_MARGIN * lmp_weight),
    analysis["forecast_mwh"]
)

# ── Compute MWOL for each strategy ───────────────────────────────────────────
def mwol_from_series(bid, actual, lmp):
    """Vectorised MWOL: sum of max(0, lmp × (bid - actual))."""
    return np.maximum(0, lmp * (bid - actual)).sum()

def revenue_from_series(bid, lmp):
    """Revenue from committed bids at positive LMP."""
    return np.maximum(0, lmp * bid).sum()

def potential_revenue(actual, lmp):
    """Maximum revenue if forecast = actual exactly."""
    return np.maximum(0, lmp * actual).sum()

act  = analysis["actual_mwh"].values
lmp  = analysis["lmp"].values
pot  = potential_revenue(act, lmp)

strategies = {
    "Raw forecast":         analysis["bid_raw"].values,
    "Conservative (−10%)":  analysis["bid_conservative"].values,
    "LMP-proportional":     analysis["bid_lmp_aware"].values,
}

print(f"\nPerfect forecast revenue: ${pot:,.2f}")
print(f"\n{'Strategy':<25} {'MWOL ($)':>12} {'Revenue ($)':>13} {'FE%':>8} {'Saved vs Raw':>14}")
print("-" * 80)

raw_mwol = None
strat_results = {}
for name, bid in strategies.items():
    mwol_val = mwol_from_series(bid, act, lmp)
    rev_val  = revenue_from_series(bid, lmp)
    fe_val   = (1 - mwol_val / pot) * 100 if pot > 0 else 0
    saved    = (raw_mwol - mwol_val) if raw_mwol is not None else 0
    strat_results[name] = {"mwol": mwol_val, "revenue": rev_val,
                            "fe": fe_val, "saved": saved}
    if raw_mwol is None:
        raw_mwol = mwol_val
    print(f"  {name:<23} ${mwol_val:>10,.2f}  ${rev_val:>11,.2f}  {fe_val:>7.2f}%  "
          f"${saved:>11,.2f}")

print("-" * 80)
best_strat = min(strat_results, key=lambda k: strat_results[k]["mwol"])
print(f"\nBest bidding strategy: '{best_strat}'")
print(f"Penalty reduction vs raw forecast: "
      f"${strat_results[best_strat]['saved']:,.2f} "
      f"({strat_results[best_strat]['saved']/raw_mwol*100:.1f}% reduction)")


In [ ]:
# ── Plot 1: MWOL breakdown by LMP regime ─────────────────────────────────────
pos_mask = analysis["lmp"] > 0
neg_mask = analysis["lmp"] <= 0
over_pos  = analysis[pos_mask &  analysis["over_predicted"]]
under_neg = analysis[neg_mask & ~analysis["over_predicted"]]

fig, ax = plt.subplots(figsize=(18, 8))
labels = ["Over-predict\n(LMP > 0)\n[expensive]",
          "Under-predict\n(LMP > 0)\n[no penalty]",
          "Over-predict\n(LMP ≤ 0)\n[no penalty]",
          "Under-predict\n(LMP ≤ 0)\n[expensive]"]
counts = [
    (pos_mask &  analysis["over_predicted"]).sum(),
    (pos_mask & ~analysis["over_predicted"]).sum(),
    (neg_mask &  analysis["over_predicted"]).sum(),
    (neg_mask & ~analysis["over_predicted"]).sum(),
]
penalties = [
    np.maximum(0, analysis.loc[pos_mask &  analysis["over_predicted"], "lmp"] *
               analysis.loc[pos_mask &  analysis["over_predicted"], "error_mwh"]).sum(),
    0,
    0,
    np.maximum(0, analysis.loc[neg_mask & ~analysis["over_predicted"], "lmp"].abs() *
               analysis.loc[neg_mask & ~analysis["over_predicted"], "error_mwh"].abs()).sum(),
]
colors_bar = ["#FF5722","#4CAF50","#4CAF50","#FF5722"]
bars = ax.bar(labels, penalties, color=colors_bar, alpha=0.85, edgecolor="white", width=0.5)
ax.bar_label(bars, labels=[f"${v:,.0f}\n({c} hrs)" for v,c in zip(penalties,counts)],
             fontsize=15, padding=5, fontweight="bold")
ax.set_title("Penalty Cost Breakdown by LMP Regime and Forecast Direction\n"
             "Only two of four quadrants incur penalties", pad=14)
ax.set_ylabel("Total Penalty Cost ($)")
ax.set_ylim(0, max(penalties)*1.35)
plt.tight_layout()
plt.savefig("figP1_penalty_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figP1_penalty_breakdown.png")

# ── Plot 2: Strategy comparison bar ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(22, 8))

strat_names = list(strat_results.keys())
mwol_vals_s = [strat_results[s]["mwol"]    for s in strat_names]
fe_vals_s   = [strat_results[s]["fe"]      for s in strat_names]
colors_s    = ["#607D8B","#2196F3","#4CAF50"]

# MWOL
bars1 = axes[0].bar(strat_names, mwol_vals_s, color=colors_s, alpha=0.88,
                     edgecolor="white", width=0.5)
axes[0].bar_label(bars1, labels=[f"${v:,.0f}" for v in mwol_vals_s],
                   fontsize=15, padding=5, fontweight="bold")
axes[0].set_title("MWOL by Bidding Strategy\nlower is better", pad=12)
axes[0].set_ylabel("Penalty Cost ($)")
axes[0].set_ylim(0, max(mwol_vals_s)*1.25)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=20, ha="right")

# FE%
bars2 = axes[1].bar(strat_names, fe_vals_s, color=colors_s, alpha=0.88,
                     edgecolor="white", width=0.5)
axes[1].bar_label(bars2, labels=[f"{v:.2f}%" for v in fe_vals_s],
                   fontsize=15, padding=5, fontweight="bold")
axes[1].axhline(100, color="black", linewidth=1.2, linestyle="--", alpha=0.3)
axes[1].set_title("Financial Efficiency by Bidding Strategy\nhigher is better", pad=12)
axes[1].set_ylabel("Financial Efficiency (%)")
axes[1].set_ylim(min(fe_vals_s)-2, 102)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha="right")

plt.suptitle(f"Bidding Strategy Comparison — {best_ml} Forecast Base",
             fontsize=20, fontweight="bold")
plt.tight_layout()
plt.savefig("figP2_strategy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figP2_strategy_comparison.png")

# ── Plot 3: Extended margin sweep with NET REVENUE (MWOL + foregone revenue) ──
# The previous sweep only minimised MWOL, hitting the 30% boundary because
# MWOL keeps falling as we bid lower — but bidding too low also LOSES revenue
# from power we produced but didn't commit to sell.
#
# TRUE OPTIMUM minimises: MWOL + foregone_revenue
#   where foregone_revenue = max(0, LMP × (actual - bid))  [missed sales]
# This creates a U-shape: too high → over-prediction penalties,
#                          too low  → missed revenue
# Extended sweep to 60% to find the real minimum.

margins = np.linspace(0, 0.60, 121)   # 0–60% in 0.5% steps

mwol_sweep_fixed      = []
foregone_sweep_fixed  = []
net_cost_sweep_fixed  = []

mwol_sweep_lmp        = []
foregone_sweep_lmp    = []
net_cost_sweep_lmp    = []

lmp_w_base = (analysis["lmp"].clip(lower=0) / max(lmp_75th, 1)).clip(0, 1).values

for m in margins:
    # Fixed margin strategy
    bid_f = np.where(analysis["lmp_pos"].values,
                     analysis["forecast_mwh"].values * (1 - m),
                     analysis["forecast_mwh"].values)
    # LMP-proportional strategy
    bid_l = np.where(analysis["lmp_pos"].values,
                     analysis["forecast_mwh"].values * (1 - m * lmp_w_base),
                     analysis["forecast_mwh"].values)

    for bid, mwol_list, foregone_list, net_list in [
        (bid_f, mwol_sweep_fixed, foregone_sweep_fixed, net_cost_sweep_fixed),
        (bid_l, mwol_sweep_lmp,   foregone_sweep_lmp,   net_cost_sweep_lmp),
    ]:
        mwol_val     = mwol_from_series(bid, act, lmp)
        # Foregone revenue: when bid < actual and LMP > 0, we sold less than produced
        foregone_val = np.maximum(0, lmp * (act - bid)).sum()
        mwol_list.append(mwol_val)
        foregone_list.append(foregone_val)
        net_list.append(mwol_val + foregone_val)

# True optimal: minimise MWOL + foregone revenue
best_m_fixed_net = margins[np.argmin(net_cost_sweep_fixed)]
best_m_lmp_net   = margins[np.argmin(net_cost_sweep_lmp)]
best_m_fixed_mwol = margins[np.argmin(mwol_sweep_fixed)]
best_m_lmp_mwol   = margins[np.argmin(mwol_sweep_lmp)]

# ── Fig P3a: Net cost curve (MWOL + foregone) ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(26, 9))

# Left: Component breakdown for fixed strategy
axes[0].plot(margins*100, mwol_sweep_fixed,     color="#FF5722", linewidth=2.5,
             label="MWOL (over-prediction penalty)")
axes[0].plot(margins*100, foregone_sweep_fixed, color="#2196F3", linewidth=2.5,
             linestyle="--", label="Foregone revenue (under-bidding cost)")
axes[0].plot(margins*100, net_cost_sweep_fixed,  color="#4CAF50", linewidth=3,
             label="Net cost (MWOL + foregone)")
axes[0].axvline(best_m_fixed_net*100, color="#4CAF50", linewidth=2, linestyle=":",
                label=f"True optimum = {best_m_fixed_net*100:.1f}%  "
                      f"net=${min(net_cost_sweep_fixed):,.0f}")
axes[0].axhline(raw_mwol, color="grey", linewidth=1.2, linestyle="-.",
                alpha=0.5, label=f"Raw forecast MWOL = ${raw_mwol:,.0f}")
axes[0].set_title("Fixed Margin — Full Cost Picture\n"
                  "MWOL falls but foregone revenue rises → true minimum exists",
                  pad=12)
axes[0].set_xlabel("Safety Margin (%)")
axes[0].set_ylabel("Cost ($)")
axes[0].legend(fontsize=14, framealpha=0.5)

# Right: LMP-proportional strategy
axes[1].plot(margins*100, mwol_sweep_lmp,     color="#FF5722", linewidth=2.5,
             label="MWOL (over-prediction penalty)")
axes[1].plot(margins*100, foregone_sweep_lmp, color="#2196F3", linewidth=2.5,
             linestyle="--", label="Foregone revenue (under-bidding cost)")
axes[1].plot(margins*100, net_cost_sweep_lmp,  color="#4CAF50", linewidth=3,
             label="Net cost (MWOL + foregone)")
axes[1].axvline(best_m_lmp_net*100, color="#4CAF50", linewidth=2, linestyle=":",
                label=f"True optimum = {best_m_lmp_net*100:.1f}%  "
                      f"net=${min(net_cost_sweep_lmp):,.0f}")
axes[1].axhline(raw_mwol, color="grey", linewidth=1.2, linestyle="-.",
                alpha=0.5, label=f"Raw forecast MWOL = ${raw_mwol:,.0f}")
axes[1].set_title("LMP-Proportional Margin — Full Cost Picture", pad=12)
axes[1].set_xlabel("Safety Margin (%)")
axes[1].set_ylabel("Cost ($)")
axes[1].legend(fontsize=14, framealpha=0.5)

plt.suptitle("Finding the TRUE Optimal Bid Discount\n"
             "Balancing over-prediction penalty vs foregone revenue",
             fontsize=20, fontweight="bold")
plt.tight_layout()
plt.savefig("figP3a_net_cost_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figP3a_net_cost_sweep.png")

# ── Fig P3b: Side-by-side comparison at optimal margins ───────────────────────
fig, ax = plt.subplots(figsize=(20, 9))
ax.plot(margins*100, net_cost_sweep_fixed, color="#2196F3", linewidth=2.5,
        label=f"Fixed margin  (optimum={best_m_fixed_net*100:.1f}%  net=${min(net_cost_sweep_fixed):,.0f})")
ax.plot(margins*100, net_cost_sweep_lmp,   color="#4CAF50", linewidth=2.5,
        linestyle="--",
        label=f"LMP-proportional  (optimum={best_m_lmp_net*100:.1f}%  net=${min(net_cost_sweep_lmp):,.0f})")
ax.axvline(best_m_fixed_net*100, color="#2196F3", linewidth=1.8, linestyle=":", alpha=0.7)
ax.axvline(best_m_lmp_net*100,   color="#4CAF50", linewidth=1.8, linestyle=":", alpha=0.7)
ax.axhline(raw_mwol, color="#FF5722", linewidth=1.5, linestyle="-.",
           alpha=0.6, label=f"No adjustment (raw MWOL = ${raw_mwol:,.0f})")
# Shade the optimal zone
opt_lo = min(best_m_fixed_net, best_m_lmp_net) * 100
opt_hi = max(best_m_fixed_net, best_m_lmp_net) * 100
ax.axvspan(opt_lo, opt_hi, alpha=0.10, color="#4CAF50", label=f"Optimal zone {opt_lo:.0f}–{opt_hi:.0f}%")
ax.set_title("Net Cost (MWOL + Foregone Revenue) vs Safety Margin\n"
             "U-shaped curve — true optimum is NOT at 30%", pad=14)
ax.set_xlabel("Safety Margin (%)")
ax.set_ylabel("Net Cost ($)  [lower = better]")
ax.legend(fontsize=14, framealpha=0.5)
plt.tight_layout()
plt.savefig("figP3b_strategy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figP3b_strategy_comparison.png")

print(f"\n{'='*65}")
print(f"  TRUE OPTIMAL MARGINS (minimising MWOL + foregone revenue)")
print(f"{'='*65}")
print(f"  Fixed margin optimal      : {best_m_fixed_net*100:.1f}%")
print(f"    → MWOL          = ${mwol_sweep_fixed[np.argmin(net_cost_sweep_fixed)]:,.2f}")
print(f"    → Foregone rev  = ${foregone_sweep_fixed[np.argmin(net_cost_sweep_fixed)]:,.2f}")
print(f"    → Net cost      = ${min(net_cost_sweep_fixed):,.2f}  "
      f"(vs ${raw_mwol:,.2f} raw = {(raw_mwol-min(net_cost_sweep_fixed))/raw_mwol*100:.1f}% better)")
print()
print(f"  LMP-proportional optimal  : {best_m_lmp_net*100:.1f}%")
print(f"    → MWOL          = ${mwol_sweep_lmp[np.argmin(net_cost_sweep_lmp)]:,.2f}")
print(f"    → Foregone rev  = ${foregone_sweep_lmp[np.argmin(net_cost_sweep_lmp)]:,.2f}")
print(f"    → Net cost      = ${min(net_cost_sweep_lmp):,.2f}  "
      f"(vs ${raw_mwol:,.2f} raw = {(raw_mwol-min(net_cost_sweep_lmp))/raw_mwol*100:.1f}% better)")
print(f"{'='*65}")

best_m_fixed = best_m_fixed_net    # update for summary below
best_m_lmp   = best_m_lmp_net

# ── Plot 4: LMP distribution during penalty hours ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(22, 8))

# LMP histogram — positive vs negative
axes[0].hist(analysis.loc[analysis["lmp"]>0, "lmp"], bins=50,
             color="#2196F3", alpha=0.7, density=True, label="Positive LMP (over-pred risky)")
axes[0].hist(analysis.loc[analysis["lmp"]<=0,"lmp"], bins=20,
             color="#FF5722", alpha=0.7, density=True, label="Negative LMP (under-pred risky)")
axes[0].axvline(0, color="black", linewidth=1.5, linestyle="--")
axes[0].set_title("LMP Distribution — Test Period\nPositive hours dominate", pad=12)
axes[0].set_xlabel("LMP ($/MWh)")
axes[0].set_ylabel("Density")
axes[0].legend(fontsize=14)

# Scatter: LMP vs forecast error — colour by penalty magnitude
penalty_scatter = np.maximum(0, analysis["lmp"] * analysis["error_mwh"])
sc = axes[1].scatter(analysis["lmp"], analysis["error_mwh"],
                      c=penalty_scatter, cmap="Reds", s=8, alpha=0.4,
                      vmin=0, vmax=penalty_scatter.quantile(0.95))
cbar = plt.colorbar(sc, ax=axes[1])
cbar.set_label("Penalty ($)", fontsize=16)
axes[1].axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.5)
axes[1].axvline(0, color="black", linewidth=1, linestyle="--", alpha=0.5)
axes[1].set_title("LMP vs Forecast Error\nRed = high penalty hours", pad=12)
axes[1].set_xlabel("LMP ($/MWh)")
axes[1].set_ylabel("Forecast Error (MWh)\n(positive = over-predicted)")
axes[1].text(lmp_75th*0.6, analysis["error_mwh"].max()*0.85,
             "⚠ High cost zone\n(over-predict + high LMP)",
             fontsize=13, color="#FF5722", ha="center")
plt.tight_layout()
plt.savefig("figP4_lmp_error_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figP4_lmp_error_scatter.png")

# ── Summary recommendation ────────────────────────────────────────────────────
print("\n" + "="*70)
print("  PENALTY MINIMISATION SUMMARY")
print("="*70)
print(f"  Best forecast model            : {best_ml}")
print(f"  Raw forecast MWOL              : ${raw_mwol:,.2f}")
print()
_fi = np.argmin(net_cost_sweep_fixed)
_li = np.argmin(net_cost_sweep_lmp)
print(f"  True optimal — Fixed margin    : {best_m_fixed_net*100:.1f}%")
print(f"    MWOL         = ${mwol_sweep_fixed[_fi]:,.2f}")
print(f"    Foregone rev = ${foregone_sweep_fixed[_fi]:,.2f}")
print(f"    Net cost     = ${net_cost_sweep_fixed[_fi]:,.2f}  "
      f"({(raw_mwol-net_cost_sweep_fixed[_fi])/raw_mwol*100:.1f}% better than raw)")
print()
print(f"  True optimal — LMP-proportional: {best_m_lmp_net*100:.1f}%")
print(f"    MWOL         = ${mwol_sweep_lmp[_li]:,.2f}")
print(f"    Foregone rev = ${foregone_sweep_lmp[_li]:,.2f}")
print(f"    Net cost     = ${net_cost_sweep_lmp[_li]:,.2f}  "
      f"({(raw_mwol-net_cost_sweep_lmp[_li])/raw_mwol*100:.1f}% better than raw)")
print()
print("  KEY RECOMMENDATIONS:")
print("  1. The true optimum is NOT at maximum discount — bidding too low")
print("     sacrifices real revenue from power you actually produced.")
print(f"  2. Optimal fixed bid discount: {best_m_fixed_net*100:.1f}% below forecast")
print("     applied only during positive-LMP hours (LMP > 0).")
print("  3. Do NOT apply any discount during negative-LMP hours —")
print("     under-delivering when LMP is negative costs money.")
print("  4. Monitor cloud cover change rate (cloud_change_1h feature).")
print("     Cloud events >20pp/hour are the main source of over-prediction.")
print(f"  5. Operational sweet spot: {min(best_m_fixed_net,best_m_lmp_net)*100:.0f}–"
      f"{max(best_m_fixed_net,best_m_lmp_net)*100:.0f}% bid discount during")
print("     positive-LMP hours balances penalty vs foregone revenue.")
print("="*70)
